# Customer Acquisition Analysis

## Purpose

This analysis showcases my approach to tackling a analysis task of comparing different user cohorts, in an effort to find the best user acquisition channel. The aim is to demonstrate an approach, which allows for detecting subtle differences between the user cohorts, allows for forecasting the eventual value of each cohort, and delivers tangible value, as the more valuable channel can be identified and levaraged early


## Premise

This analysis assumes a digital product or service, and focuses on user acquisition through digital channels where user attribution is straightforward. The basic premise of this analysis is best described as freemium game or other similar entertainment produc/service, but the approach can be applied with slight modifications to e.g. e-commerce shopping, professional use software, etc.

Essentially the business model should rely on acquiring user, who will then engange with the product, and spend money on in app purchases, or purchase licenses or subscriptions, or buy products from shop. Different kind of users will have a different spending profile, which will be more or less linked to their other behavior on the platform.

It is also assumed, that there is historical data from which the spending pattern of different user clusters can be reasonably well projected. This assumes that the product has history and the business is not treading a new path, which would be case in e.g. after launching completely new product or vertical, entering a vastly different new market, or experiencing substantial growth and expanding outside of a niche market into more general market position. Also it is assumed that there is historical data where the fundamental way of users engaging with the product and spending money is similar to current and near future situation. This means that no comprehensive changes in e.g. pricing, product structure, or business logic has been made. In case of a game or similar entertainment product it is assumed that there has not been substantial changes in game design or other engagement factors, which would alter the way how users both act when using the product, and how that activity is linked to spending profile. In short, for the purpose of this excercise, it is possible to draw valid conclusions about future based on historical data as far as user activity and spending predictions are concerned.


## Task

There is ongoing intiative comparing 2 user acquisition channels. The chosen metric to compare these 2 channels is the net value per user at 90 days after install. That is calculated as 90d Lifetime value per user - Cost per Acquisition.

The cutoff point of the comparison is chosen at 90 days, since after that there are too many uncertainties to warrant the assumption that the user behavior and value would be driven primarily by the user acquisition channel through which they have been acquired. After that the overall experience on the platform, possible reactivation campaigns, or changes in personal situation will likely influence the predicted value of the user more than the user acquisition channel. Moreover, there might be other changes in e.g. product and/or business logic, game design, or meta in case there is multiplayer component to the product.

The specific metrics will be properly displayed and visualized in the analysis proper part, but a preliminary situation is that the campaign has ran for so long that there is 30 days worth of user activity data for about 10000 acquired users in both channels. The CPA for both channels is roughly similar, and so is the 30 day LTV per user. However, there is a sense that there might be some critical differences in the profitability potential between the 2 cohorts. The task is to conclusively decide which channel to discontinue in order to leverage the more profitable channel as early as possible, to minimize the opprtunity cost incurred by running inefficient user acquisition channels.


## Disclaimers

While the scenario, high level figures, and other realities of this excercise are based on the real world experiences of the creator of this analysis example, the data is purely made up for the purpose of demonstration. This analysis or the dataset used here does not yield any strategic or operative insights on the businesses of any previous or current employer of the creator, nor any other real world company with which the creator might hav worked with through his career.

This also means that the data used in this excercise is simplified in nature compared to real world user data. The asusmptions in this excercise, e.g. boundaries between different user clusters, or the predictability of the user spending profiles, or the link between 30 day user activity and 90 day spending profile might not be as clear in real life as presented here. Thus, this analysis should not be taken as "ready to implement turnkey solution" to solving real world user acquisition value prediction tasks, but more as an example of the approach and knowledge basis of the creator of this example. In real world the actual solutions will be built on top of this foundation case by case, with modifications and additions to the approach applied as the situation necessitates.

Moreover, the creator is first and foremost a full stack data analyst, whose expertise lies in designing, building, and executing analysis pipelines, from problem statement and metric definition, through data pipeline and analysis tool development, to presenting and visualizing the results in a way which drives decision making creating measurable business impact and value. This means that the creator is not professional python developer. The creator's philosophy is to use python as tool to get things done and deliver impactful business insights. Thus, the code might contain unoptimized parts, or occasionally an approach is taken, which is considered to not adhere to the best practices by professional python developer community. When applying this (or similar approach) in production, the creator would prefer to build the analysis by using SQL and a professional BI tool, and would refine many parts of this code if the application is necessary to do with python.


# Code Setup

## Import libraries

In [1]:
from IPython.display import clear_output, display, Markdown, HTML
import pandas as pd
import plotly.express as px
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import lognorm
import random
import time

import os
import dotenv
dotenv.load_dotenv()

import psycopg

from sqlalchemy import create_engine

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity, cosine_distances
import hdbscan

## Database connection

In [2]:
def postgres_connection():

    username = os.getenv('USER')
    pwd = os.getenv('PWD')
    host = os.getenv('HOST')
    port = os.getenv('PORT')
    default_db = os.getenv('DEFAULT_DB')

    conn = psycopg.connect(
        host = host,
        dbname = default_db,
        user = username,
        password = pwd,
        port = port
    )
    return conn


def select_table(query: str) -> pd.DataFrame:
    conn = postgres_connection()
    with conn:
        with conn.cursor() as cur:
            cur.execute(query)
            column_headers = [desc[0] for desc in cur.description]
            df = pd.DataFrame(cur.fetchall(), columns=column_headers)
            conn.commit()
            cur.close()
    return df


def execute_sql(query: str):
    conn = postgres_connection()
    with conn:
        with conn.cursor() as cur:
            cur.execute(query)
            print(f"PostgreSQL response: {cur.statusmessage}")

    conn.close()


def postgres_connection_string():
    """
    Create a connection string for SQLAlchemy to connect to PostgreSQL.
    """
    username = os.getenv("USER")
    pwd = os.getenv("PWD")
    host = os.getenv("HOST")
    port = os.getenv("PORT", 5432)
    default_db = os.getenv("DEFAULT_DB")

    return f"postgresql+psycopg://{username}:{pwd}@{host}:{port}/{default_db}"


def create_table_from_df(df, table_name):
    """
    Create a PostgreSQL table using SQLAlchemy and DataFrame structure.
    """
    connection_string = postgres_connection_string()
    engine = create_engine(connection_string)

    with engine.connect() as conn:
        df.head(0).to_sql(table_name, conn, if_exists="replace", index=False)
        print(f"Table '{table_name}' created successfully.")


def insert_df_to_table(df, table_name):
    """
    Insert rows from a pandas DataFrame into a PostgreSQL table using SQLAlchemy.
    """
    connection_string = postgres_connection_string()
    engine = create_engine(connection_string)

    with engine.connect() as conn:
        df.to_sql(table_name, conn, if_exists="append", index=False)
        print(f"Data inserted successfully into table '{table_name}'.")


## Data visualization

In [ ]:
def hdbscan_clustering(df, model_input, min_cluster_size):
  model = SentenceTransformer(model_input)
  ### New
  embeddings = model.encode(df['text_clustering'].tolist())
  ### HDBSCAN
  clusterer = hdbscan.HDBSCAN(min_cluster_size)
  clusters = clusterer.fit_predict(embeddings)
  sentences = df['text_clustering'].tolist()
  #pages = df['page'].tolist()
  #sources = df['source_file'].tolist()
  identifiers = df['identifier_clustering'].tolist()
  try:
    ratings = df['rating'].tolist()
  except KeyError:
    print('No rating, placeholder values instead')
    ratings = [1] * len(df)
  # Create DataFrame for analysis
  #df_clustered = pd.DataFrame({'page': pages, 'source': source_file, 'text': sentences, 'cluster': clusters, 'identifier': identifiers})
  df_clustered = pd.DataFrame({'text': sentences, 'cluster': clusters, 'identifier': identifiers, 'rating': ratings})
  return df_clustered, embeddings

def kmeans_clustering(df, num_clusters, model_input):
  model = SentenceTransformer(model_input)
  keywords = df['text_clustering'].tolist()
  embeddings = model.encode(keywords)
  kmeans = KMeans(n_clusters=num_clusters, random_state=0).fit(embeddings)
  clusters = kmeans.labels_
  sentences = df['text_clustering'].tolist()
  identifiers = df['identifier_clustering'].tolist()
  try:
    ratings = df['rating'].tolist()
  except KeyError:
    print('No rating, placeholder values instead')
    ratings = [1] * len(df)
  # Create DataFrame for analysis
  df_clustered = pd.DataFrame({'text': sentences, 'cluster': clusters, 'identifier': identifiers, 'rating': ratings})
  return df_clustered, embeddings

def scatter_with_clusters(df_clustered, embeddings, visual_dims, rating_coloring):
  clusters = df_clustered['cluster'].tolist()
  if visual_dims == 2:
    pca = PCA(n_components=2)
    reduced_embeddings = pca.fit_transform(embeddings)
    data_for_plot = pd.DataFrame({
        'x': reduced_embeddings[:, 0],
        'y': reduced_embeddings[:, 1],
        'Cluster': clusters,
        'Identifier': df_clustered['identifier']
    })
    fig = px.scatter(data_for_plot, x='x', y='y', color='Cluster', hover_name='Identifier', title='KMeans Clustering Visualization')
    fig.update_traces(marker=dict(size=8), selector=dict(mode='markers'))
    fig.show()
  elif visual_dims == 3:
    pca = PCA(n_components=3)
    #print(4)
    reduced_embeddings = pca.fit_transform(embeddings)
    data_for_plot = pd.DataFrame({
        'x': reduced_embeddings[:, 0],
        'y': reduced_embeddings[:, 1],
        'z': reduced_embeddings[:, 2],
        'Cluster': clusters,
        'Identifier': df_clustered['identifier'],
        'Rating': df_clustered['rating']
    })
    #print(5)
    if rating_coloring:
      coloring = 'Rating'
    else:
      coloring = 'Cluster'
    fig = px.scatter_3d(data_for_plot, x='x', y='y', z='z',
                        color=coloring, hover_name='Identifier',
                        title='Clustering Visualization',
                        width=800, height=800,
                        color_continuous_scale='Spectral',  # Adjust the color scale as needed
                        color_discrete_sequence=px.colors.qualitative.Set1)
    fig.update_traces(marker=dict(size=8), selector=dict(mode='markers'))
    fig.show()
  else:
    print('Please input 2 or 3 dimensions for visualization')
    pass
  return data_for_plot

def check_cols(df, colname):
    retry = True
    while retry:
        try:
            test = df[f'{colname}'].iloc[0]
            retry = False
        except KeyError:
            print(df.columns.tolist())
            id_clustering = input(f'Select one of the cols above to be data point {colname}: ')
            try:
                df[f'{colname}'] = df[f'{id_clustering}']
            except KeyError:
                print('Invalid column name')
    return df


## Other

In [9]:
def display_scrollable(df, max_rows=999):
    pd.options.display.max_rows = 999
    pd.options.display.max_columns = 999
    html = df.to_html(max_rows=max_rows)
    html = f'<div style="max-height: 500px; overflow-y: scroll;">{html}</div>'
    display(HTML(html))
    pd.options.display.max_rows = 15
    pd.options.display.max_columns = 15


# Analysis

In [10]:
query_text = '''
with

base as (
    select
        user_id::text,
        session_id,
        session_start,
        session_length,
        session_end,
        action_id,
        transaction_size,
        running_session_number
    from fct_user_action_balanced
)
select * from base
'''


df = select_table(query_text)
display_scrollable(df)

,user_id,session_id,session_start,session_length,session_end,action_id,transaction_size,running_session_number
0,0,None,NaT,NaN,NaT,install,NaN,NaN
1,0,0-0,2024-01-08 19:01:22.000000,1981.583279,2024-01-08 19:34:23.583279,session_start,NaN,0.0
2,0,0-1,2024-01-09 17:21:24.943262,1369.461513,2024-01-09 17:44:14.404775,session_start,NaN,1.0
3,0,0-2,2024-01-10 17:50:16.569231,1536.371197,2024-01-10 18:15:52.940428,session_start,NaN,2.0
4,0,0-3,2024-01-11 07:49:18.282674,680.872667,2024-01-11 08:00:39.155341,session_start,NaN,3.0
5,0,0-4,2024-01-12 08:06:48.171288,1358.328583,2024-01-12 08:29:26.499870,session_start,NaN,4.0
6,0,0-5,2024-01-13 15:44:17.720786,1479.058339,2024-01-13 16:08:56.779125,session_start,NaN,5.0
7,0,0-6,2024-01-14 15:27:26.721052,891.357360,2024-01-14 15:42:18.078412,session_start,NaN,6.0
8,0,0-7,2024-01-15 23:35:26.359292,473.462030,2024-01-15 23:43:19.821322,session_start,NaN,7.0
9,0,0-8,2024-01-16 22:07:48.401374,754.729300,2024-01-16 22:20:23.130674,session_start,NaN,8.0


In [5]:
execute_sql('''DROP TABLE "fct_user_action_b-heavy"''')

PostgreSQL response: DROP TABLE
